# 共享 Tile 与线程间同步

SIMT 函数内能访问什么、怎么访问、何时要等——本节讲清数据访问与同步的全部规则。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 四类对象的访问方式

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">对象</th><th style="text-align:left;">访问方式</th><th style="text-align:left;">要求</th></tr>
<tr><td>Scalar</td><td>局部变量、算术/比较/布尔表达式、simt 标量接口</td><td>每线程独立计算；数学函数操作数 dtype 一致</td></tr>
<tr><td>GM Tensor</td><td><code>tensor[i, j]</code> 完整标量下标</td><td>非零 Rank、静态 Shape、ND 布局；索引运行时可算，有效性自保证</td></tr>
<tr><td>UB Tile</td><td><code>tile[row, col]</code></td><td>静态二维 Shape、UB、ND；由外层 Kernel 创建管理</td></tr>
<tr><td>Tile 有效区域</td><td><code>tile.valid_shape[0] / [1]</code></td><td>读运行期有效行列数，尾块按下标保护</td></tr>
</table>

限制：Tensor/Tile 须以**完整对象**传入（不支持 Slice/Subview 作 Tensor/Tile 参数；元素读取所得 Scalar 可以传给 Scalar 参数）；元素位宽 ≥ 8bit；**运行期索引 ≠ 动态 Tensor Shape（后者不支持）**。


## 2. 边界保护：你的责任

SIMT 对下标的校验分两档：

- **静态常量下标**：前端按 Shape 检查，越界报错（如 [1,64] 时 `a[0,64]` 编译期报错）；
- **动态下标**（来自线程编号/入参/表达式）：**前端不检查，开发者保证**——读写前显式加 `0 <= index < Shape[axis]` 判断。

访问 Tile 还须保证下标 < valid_shape（有效区域由外层设置传入，SIMT 不自动限制）：

```python
@pl.vector_function(mode="simt", max_threads=256)
def add_valid(data, delta: pl.DT_FP32):
    row = pl.simt.thread_idx().y
    col = pl.simt.thread_idx().x
    if row < data.valid_shape[0] and col < data.valid_shape[1]:
        data[row, col] = data[row, col] + delta
```

【待 950 验证：官方边界保护示例】

越界读写的后果：破坏相邻数据或执行异常——**需要在索引设计时排除，而不能依赖编译器兜底**。


## 3. 块内同步：syncthreads

线程间通过**共享 Tile** 协作时，用 `pl.simt.syncthreads()` 做块内屏障——所有线程到达后继续，保证屏障前的写入对屏障后的读取可见。三条纪律：

1. **块内所有参与线程在一致的动态次数到达**屏障；本课程将屏障放在无条件路径，避免分歧分支或提前 return 导致部分线程缺席；
2. 该屏障只同步**本 Block 内**线程，不提供跨 Block 同步（回顾 softmax_simt 示例：三个 syncthreads 全部无条件下达）；
3. 归约状态（最大值/累加和）放共享 Tile，"单线程初始化 → 屏障 → 全员原子累加 → 屏障 → 全员读"是标准协作模式。


## 4. 外层流水同步

SIMT 入口函数在 **V 流水**异步执行，与搬运的依赖：

```text
load（MTE2）→ MTE2/V 同步 → SIMT/SIMD 计算（V）→ V/MTE3 同步 → store（MTE3）
```

用带 mutex_ids 的 make_tile_group + auto_mutex=True 时，框架自动管理 load/SIMT 调用/store 的缓冲依赖；普通 make_tile 需手动 sync_src/sync_dst。


### 完整块内计数：每个 Block 有自己的 32 B 共享区

为计数器声明 [1,8] INT32 Tile，满足 32 B 物理对齐，只使用 [0,0]。每个 Vector 核拥有独立 UB，因此相同片上地址不表示多个 Block 共用一个计数器。线程 0 初始化；所有线程经过第一道屏障后各加 1；第二道屏障后线程 0 才把最终值写入 GM 的本 Block 位置。

两个屏障都在无条件路径上。第一道保护“初始化完成”，第二道保护“全部累加完成”；原子只保证一次更新不丢失，不能替代任一道阶段边界。


In [ ]:
@pl.vector_function(mode="simt",max_threads=256)
def count_threads(shared,out):
    tid = pl.simt.thread_idx().x
    if tid == 0:
        shared[0,0] = 0
    pl.simt.syncthreads()
    pl.simt.atomic_add(shared, [0,0], 1)
    pl.simt.syncthreads()
    if tid == 0:
        out[0,pl.simt.block_idx().x] = shared[0,0]


In [ ]:
@pl.jit(auto_mutex=True)
def count_kernel(out: pl.Tensor[[1,4],pl.DT_INT32]):
    tt = pl.TileType(shape=[1,8],dtype=pl.DT_INT32,target_memory=pl.MemorySpace.Vec)
    group = pl.make_tile_group(type=tt,addrs=0,mutex_ids=[0])
    with pl.section_vector():
        shared = group.current()
        count_threads[256](shared,out)


In [ ]:
device=f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
out=torch.full((1,4),-1,device=device,dtype=torch.int32)
count_kernel[None,4](out)
torch.npu.synchronize()
torch.testing.assert_close(out,torch.full_like(out,256),rtol=0,atol=0)
print('四个 Block 的计数均为 256')


## 5. 课后练习

1. （判断）动态下标 `a[i, j]` 越界时前端会报错。（对/错）
2. （填空）`pl.simt.syncthreads()` 同步的范围是 ______ 内的线程；把屏障放进 if 分支的风险是 ______。
3. （实践）把 softmax_simt 的协作模式抽象成五步（初始化→屏障→原子累加→屏障→读取），用一个"块内计数"例子（每线程 +1，块内求和）套用这个模式写出伪代码。

In [ ]:
!cat ./answer/06.04_answer.txt

## 6. 本节小结

- 四类对象访问规则表（完整对象传入、≥8bit、动态 Shape 不支持）；
- **动态下标有效性是开发者责任**：显式边界判断 + valid_shape 保护；
- syncthreads 块内屏障：无条件下达、只管本 Block；
- SIMT 在 V 流水异步执行，与搬运的依赖靠 auto_mutex 或手动同步。